
# Treinamento de Modelos de Detecção de Pneumonia em Radiografias de Tórax

Este notebook treina **diversas arquiteturas de CNN**, cada uma otimizada para
um objetivo diferente, seguindo o mesmo padrão de código do protótipo original
(`Sequential` + `Rescaling` + blocos `Conv2D`/`MaxPooling2D`/`Dropout` + cabeça
densa com `sigmoid`):

| Modelo | Objetivo | Ideia da arquitetura |
|---|---|---|
| `modelo_alta_acuracia` | Melhor **acurácia geral** | CNN de profundidade média, mapa final 14×14 |
| `modelo_alta_precisao` | Melhor **precisão** (menos falsos positivos) | CNN mais profunda, mais regularização (Dropout + L2), decisão mais conservadora |
| `modelo_alto_f1` | Melhor **F1-score** (equilíbrio precisão/recall) | CNN com `BatchNormalization` e `GlobalAveragePooling2D`, mais estável no treino |
| `modelo_leve_rapido` | **Inferência rápida** (bônus) | CNN mais rasa, poucos parâmetros, ótima para triagem em tempo real |

Ao final, os 3 primeiros modelos correspondem diretamente aos modelos
cadastrados em `MODEL_CATALOG` da interface Streamlit
(`ModeloA_AltaAcuracia`, `ModeloB_AltaPrecisao`, `ModeloC_AltoF1`).

> **Dataset esperado:** estrutura de pastas no padrão do "Chest X-Ray Images
> (Pneumonia)":
> ```
> BASE_DIR/
> ├── train/NORMAL, train/PNEUMONIA
> ├── val/NORMAL,   val/PNEUMONIA
> └── test/NORMAL,  test/PNEUMONIA
> ```
> Ajuste `BASE_DIR` na célula de configuração para o caminho do seu dataset.


## 1. Imports

In [ ]:

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Rescaling, Conv2D, MaxPooling2D, Dropout, Flatten, Dense,
    BatchNormalization, GlobalAveragePooling2D,
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
    precision_recall_curve, classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)


## 2. Configurações gerais e carregamento dos dados

In [ ]:

BASE_DIR = "data"          # <-- ajuste para o caminho do seu dataset
IMG_SIZE = (252, 252)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

class_names = train_ds.class_names  # ex.: ['NORMAL', 'PNEUMONIA']
print("Classes:", class_names)

# Cache e prefetch para acelerar o treinamento
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:

# O dataset de pneumonia costuma ser desbalanceado (mais casos de PNEUMONIA
# do que NORMAL). Calculamos pesos de classe para compensar isso no treino.
train_labels = np.concatenate([y.numpy() for _, y in train_ds.unbatch().batch(1024)])
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels.ravel(),
)
class_weight = {i: w for i, w in enumerate(class_weights_array)}
print("Pesos de classe:", class_weight, "->", dict(zip(class_names, class_weights_array)))



## 3. Arquiteturas dos modelos

### 3.1 `modelo_alta_acuracia`
CNN de profundidade média (3 blocos convolucionais), reduzindo o mapa de
características até **14×14**. Segue exatamente o padrão do protótipo
original — bom equilíbrio geral entre viés e variância.


In [ ]:

modelo_alta_acuracia = Sequential([
    Rescaling(1./255, input_shape=(252, 252, 3)),   # 252 x 252

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((3, 3)),                            # -> 126 x 126 (aprox.)
    Dropout(0.2),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((3, 3)),                            # -> 42 x 42
    Dropout(0.2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((3, 3)),                            # -> 14 x 14

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
], name="modelo_alta_acuracia")



### 3.2 `modelo_alta_precisao`
CNN mais profunda (4 blocos), com **mais filtros**, **Dropout mais agressivo**
e **regularização L2** nas camadas densas. O objetivo é uma fronteira de
decisão mais conservadora — o modelo só "aposta" em Pneumonia quando está
bastante confiante, reduzindo falsos positivos (maior precisão).


In [ ]:

L2 = 1e-4

modelo_alta_precisao = Sequential([
    Rescaling(1./255, input_shape=(252, 252, 3)),    # 252 x 252

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((3, 3)),                             # -> 126 x 126
    Dropout(0.3),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((3, 3)),                             # -> 42 x 42
    Dropout(0.3),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((3, 3)),                             # -> 14 x 14
    Dropout(0.3),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),                             # -> 7 x 7
    Dropout(0.4),

    Flatten(),

    Dense(256, activation="relu", kernel_regularizer=l2(L2)),
    Dropout(0.4),
    Dense(256, activation="relu", kernel_regularizer=l2(L2)),
    Dropout(0.4),
    Dense(1, activation="sigmoid"),
], name="modelo_alta_precisao")



### 3.3 `modelo_alto_f1`
CNN com `BatchNormalization` após cada convolução (treino mais estável e
rápido) e `GlobalAveragePooling2D` no lugar de `Flatten` (menos parâmetros,
menor overfitting). É o modelo desenhado para o **melhor equilíbrio entre
precisão e recall** — bom para triagem geral, sem viés forte para nenhum dos
dois erros (falso positivo vs. falso negativo).


In [ ]:

modelo_alto_f1 = Sequential([
    Rescaling(1./255, input_shape=(252, 252, 3)),     # 252 x 252

    Conv2D(32, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),                              # -> 126 x 126
    Dropout(0.2),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),                              # -> 63 x 63
    Dropout(0.25),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),                              # -> 31 x 31
    Dropout(0.25),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),                              # -> 15 x 15
    Dropout(0.3),

    GlobalAveragePooling2D(),

    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
], name="modelo_alto_f1")



### 3.4 `modelo_leve_rapido` (bônus)
Arquitetura mais rasa e com poucos filtros — muito menos parâmetros que as
anteriores. Não é o modelo mais preciso, mas é o mais **rápido para
inferência**, útil em cenários de triagem em tempo real ou hardware limitado.


In [ ]:

modelo_leve_rapido = Sequential([
    Rescaling(1./255, input_shape=(252, 252, 3)),     # 252 x 252

    Conv2D(16, (3, 3), activation="relu"),
    MaxPooling2D((4, 4)),                              # -> 63 x 63
    Dropout(0.2),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((4, 4)),                              # -> 15 x 15
    Dropout(0.2),

    Flatten(),

    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
], name="modelo_leve_rapido")


In [ ]:

# Resumo comparativo da arquitetura de cada modelo
modelo_alta_acuracia.summary()
modelo_alta_precisao.summary()
modelo_alto_f1.summary()
modelo_leve_rapido.summary()


## 4. Compilação dos modelos

In [ ]:

def compilar(modelo):
    modelo.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.F1Score(name="f1_score"),
        ],
    )

modelos = {
    "modelo_alta_acuracia": modelo_alta_acuracia,
    "modelo_alta_precisao": modelo_alta_precisao,
    "modelo_alto_f1": modelo_alto_f1,
    "modelo_leve_rapido": modelo_leve_rapido,
}

for nome, modelo in modelos.items():
    compilar(modelo)
    print(f"{nome}: compilado.")


## 5. Callbacks de treinamento

In [ ]:

def criar_callbacks(nome_modelo):
    return [
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        ModelCheckpoint(
            filepath=f"models/{nome_modelo}.keras",
            monitor="val_loss",
            save_best_only=True,
        ),
    ]

os.makedirs("models", exist_ok=True)
EPOCHS = 30


## 6. Treinamento

In [ ]:

historicos = {}

print("Treinando modelo_alta_acuracia...")
historicos["modelo_alta_acuracia"] = modelo_alta_acuracia.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=criar_callbacks("modelo_alta_acuracia"),
    verbose=1,
)


In [ ]:

print("Treinando modelo_alta_precisao...")
historicos["modelo_alta_precisao"] = modelo_alta_precisao.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=criar_callbacks("modelo_alta_precisao"),
    verbose=1,
)


In [ ]:

print("Treinando modelo_alto_f1...")
historicos["modelo_alto_f1"] = modelo_alto_f1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=criar_callbacks("modelo_alto_f1"),
    verbose=1,
)


In [ ]:

print("Treinando modelo_leve_rapido...")
historicos["modelo_leve_rapido"] = modelo_leve_rapido.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=criar_callbacks("modelo_leve_rapido"),
    verbose=1,
)



## 7. Avaliação no conjunto de teste

Coletamos, para cada modelo: métricas agregadas (`evaluate`), as
probabilidades previstas e os rótulos verdadeiros — tudo isso será usado na
célula de gráficos logo a seguir.


In [ ]:

resultados_teste = {}

y_true = np.concatenate([y.numpy() for _, y in test_ds.unbatch().batch(1024)]).ravel()

for nome, modelo in modelos.items():
    print(f"\nAvaliando {nome}...")
    # return_dict=True garante compatibilidade entre versões do Keras:
    # em algumas versões (Keras 3), `model.metrics_names` não traz mais os
    # nomes individuais das métricas, apenas ['loss', 'compile_metrics'].
    metrics_dict = modelo.evaluate(test_ds, verbose=0, return_dict=True)

    y_prob = modelo.predict(test_ds, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    resultados_teste[nome] = {
        "metrics": metrics_dict,
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
    }

    print(f"  Acurácia:  {metrics_dict.get('accuracy', float('nan')):.4f}")
    print(f"  Precisão:  {metrics_dict.get('precision', float('nan')):.4f}")
    print(f"  Recall:    {metrics_dict.get('recall', float('nan')):.4f}")
    print(f"  F1-score:  {np.ravel(metrics_dict.get('f1_score', float('nan')))[0]:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names))



---
## 8. Gráficos de Teste

Célula dedicada exclusivamente à visualização dos resultados: curvas de
treino/validação, matrizes de confusão, curvas ROC, curvas
Precisão-Recall e comparativo final entre modelos.


In [ ]:

# ---------------------------------------------------------------------------
# 8.1 Curvas de treino/validação (acurácia e loss) — uma linha por modelo
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(len(modelos), 2, figsize=(12, 4 * len(modelos)))

for i, (nome, hist) in enumerate(historicos.items()):
    h = hist.history

    axes[i, 0].plot(h["accuracy"], label="Treino")
    axes[i, 0].plot(h["val_accuracy"], label="Validação")
    axes[i, 0].set_title(f"{nome} — Acurácia")
    axes[i, 0].set_xlabel("Época")
    axes[i, 0].set_ylabel("Acurácia")
    axes[i, 0].legend()
    axes[i, 0].grid(alpha=0.3)

    axes[i, 1].plot(h["loss"], label="Treino")
    axes[i, 1].plot(h["val_loss"], label="Validação")
    axes[i, 1].set_title(f"{nome} — Loss")
    axes[i, 1].set_xlabel("Época")
    axes[i, 1].set_ylabel("Loss")
    axes[i, 1].legend()
    axes[i, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# ---------------------------------------------------------------------------
# 8.2 Matrizes de confusão no conjunto de teste
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(modelos), figsize=(5 * len(modelos), 4.5))
if len(modelos) == 1:
    axes = [axes]

for ax, (nome, res) in zip(axes, resultados_teste.items()):
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(nome)

plt.tight_layout()
plt.show()


In [ ]:

# ---------------------------------------------------------------------------
# 8.3 Curvas ROC comparando todos os modelos
# ---------------------------------------------------------------------------
plt.figure(figsize=(7, 6))

for nome, res in resultados_teste.items():
    fpr, tpr, _ = roc_curve(res["y_true"], res["y_prob"])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{nome} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
plt.xlabel("Taxa de Falsos Positivos (1 - Especificidade)")
plt.ylabel("Taxa de Verdadeiros Positivos (Sensibilidade)")
plt.title("Curva ROC — Comparação entre Modelos")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# ---------------------------------------------------------------------------
# 8.4 Curvas Precisão-Recall comparando todos os modelos
# ---------------------------------------------------------------------------
plt.figure(figsize=(7, 6))

for nome, res in resultados_teste.items():
    precision_vals, recall_vals, _ = precision_recall_curve(res["y_true"], res["y_prob"])
    plt.plot(recall_vals, precision_vals, label=nome)

plt.xlabel("Recall (Sensibilidade)")
plt.ylabel("Precisão")
plt.title("Curva Precisão-Recall — Comparação entre Modelos")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# ---------------------------------------------------------------------------
# 8.5 Comparativo final: Acurácia, Precisão, Recall e F1-score por modelo
# ---------------------------------------------------------------------------
nomes_modelos = list(resultados_teste.keys())
metricas_labels = ["accuracy", "precision", "recall", "f1_score"]
metricas_titulos = ["Acurácia", "Precisão", "Recall (Sensibilidade)", "F1-score"]

valores = {m: [] for m in metricas_labels}
for nome in nomes_modelos:
    m = resultados_teste[nome]["metrics"]
    for chave in metricas_labels:
        v = np.ravel(m.get(chave, np.nan))[0]
        valores[chave].append(v)

x = np.arange(len(nomes_modelos))
largura = 0.2

fig, ax = plt.subplots(figsize=(10, 6))
for i, (chave, titulo) in enumerate(zip(metricas_labels, metricas_titulos)):
    ax.bar(x + i * largura, valores[chave], width=largura, label=titulo)

ax.set_xticks(x + largura * 1.5)
ax.set_xticklabels(nomes_modelos, rotation=15)
ax.set_ylabel("Valor da métrica")
ax.set_ylim(0, 1)
ax.set_title("Comparativo de Métricas no Conjunto de Teste")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Tabela resumo
print(f"{'Modelo':<22}{'Acurácia':>10}{'Precisão':>10}{'Recall':>10}{'F1-score':>10}")
for nome in nomes_modelos:
    m = resultados_teste[nome]["metrics"]
    acc = np.ravel(m.get('accuracy', np.nan))[0]
    prec = np.ravel(m.get('precision', np.nan))[0]
    rec = np.ravel(m.get('recall', np.nan))[0]
    f1 = np.ravel(m.get('f1_score', np.nan))[0]
    print(f"{nome:<22}{acc:>10.4f}{prec:>10.4f}{rec:>10.4f}{f1:>10.4f}")
